In [6]:
import pickle
import numpy as np
import pandas as pd

# 1. Load the generated samples
# Replace with your actual filename
filename = "../Data/liesel_output_margarine_paper_redo_40000_samples.pkl" 

with open(filename, "rb") as f:
    res = pickle.load(f)

samples = res["samples"]

# Define Labels
param_names = ["Blue Bonnet", "Fleischmanns", "House", "Generic", "Shed Spread", "LogPrice"]
demo_names = ["Intercept", "log(Income)", "Family size"]
k_dim = len(param_names)

# ==========================================
# REPRODUCING TABLE 5.2: Posterior of Delta
# ==========================================
# Flatten chains and draws: shape becomes (total_draws, n_z, k_dim)
delta_draws = samples["Delta"].reshape(-1, len(demo_names), k_dim)

delta_mean = np.mean(delta_draws, axis=0)
delta_std = np.std(delta_draws, axis=0)

print("=== TABLE 5.2: Posterior distribution of Delta ===")
delta_df = pd.DataFrame(index=demo_names, columns=param_names)

for i in range(len(demo_names)):
    for j in range(k_dim):
        mean_val = delta_mean[i, j]
        std_val = delta_std[i, j]
        delta_df.iloc[i, j] = f"{mean_val: .2f} ({std_val:.2f})"

display(delta_df)
print("\n")


# ==========================================
# REPRODUCING TABLE 5.1: Covariance / Correlation
# ==========================================
# L is the Cholesky factor of the precision matrix (V_inv)
L_draws = samples["sigma_inv_chol"].reshape(-1, k_dim, k_dim)
n_draws = L_draws.shape[0]

std_draws = np.zeros((n_draws, k_dim))
corr_draws = np.zeros((n_draws, k_dim, k_dim))

# Transform precision Cholesky to Covariance, Stds, and Correlations per draw
for i in range(n_draws):
    L = L_draws[i]
    
    # V_inv = L @ L^T
    precision_matrix = L @ L.T
    
    # V = V_inv^-1
    covariance_matrix = np.linalg.inv(precision_matrix)
    
    # Standard Deviations (sqrt of diagonal)
    stds = np.sqrt(np.diag(covariance_matrix))
    std_draws[i] = stds
    
    # Correlation Matrix = V / (stds * stds^T)
    outer_stds = np.outer(stds, stds)
    corr_draws[i] = covariance_matrix / outer_stds

# Calculate Posterior Means and Standard Deviations
std_mean = np.mean(std_draws, axis=0)
std_sd = np.std(std_draws, axis=0)

corr_mean = np.mean(corr_draws, axis=0)
corr_sd = np.std(corr_draws, axis=0)

print("=== TABLE 5.1: Correlations and standard deviations of betas ===")
table_5_1_df = pd.DataFrame(index=param_names, columns=param_names)

for i in range(k_dim):
    for j in range(k_dim):
        if i == j:
            # Diagonal: Standard Deviations
            mean_val = std_mean[i]
            sd_val = std_sd[i]
            table_5_1_df.iloc[i, j] = f"{mean_val:.2f} ({sd_val:.2f})"
        elif j > i:
            # Upper triangle: Correlations
            mean_val = corr_mean[i, j]
            sd_val = corr_sd[i, j]
            table_5_1_df.iloc[i, j] = f"{mean_val:.2f} ({sd_val:.2f})"
        else:
            # Lower triangle: leave blank
            table_5_1_df.iloc[i, j] = ""

display(table_5_1_df)



=== TABLE 5.2: Posterior distribution of Delta ===


,Blue Bonnet,Fleischmanns,House,Generic,Shed Spread,LogPrice
Intercept,-1.14 (0.13),-3.63 (0.76),-2.47 (0.19),-5.01 (0.31),-1.76 (0.35),-4.03 (0.17)
log(Income),0.03 (0.23),0.98 (0.72),-0.02 (0.32),-0.64 (0.42),-0.67 (0.45),-0.31 (0.30)
Family size,0.04 (0.30),-2.33 (0.94),0.75 (0.42),2.00 (0.58),0.17 (0.61),0.38 (0.39)


KeyError: 'sigma_inv_chol'